# AVTR-1 conversational avatar server

Run the single code cell below. It prepares AVTR-1, downloads/builds the required models, preloads them, and prints a temporary `gradio.live` URL. The first run asks to mount Google Drive and saves a GPU/TensorRT-specific engine cache there; compatible later runtimes restore it instead of rebuilding. The Kiosk tab begins with a silent camera mirror, then switches to a voice-cloned avatar after an 8–15 second enrollment recording. Conversation is push-to-talk and each turn includes a performance profile.

Before starting: accept the model conditions at [avaturn-live/avtr-1](https://huggingface.co/avaturn-live/avtr-1), select an **A100 or L4 GPU** runtime, and add `HF_TOKEN` to Colab Secrets. Optionally add `GRADIO_PASSWORD`; otherwise the cell generates and prints a random password. Only clone a face and voice with the person's explicit permission.

In [ ]:
# One-time setup followed by the persistent Gradio server. Keep this cell running.
import getpass, hashlib, json, os, secrets, shutil, subprocess, sys, time
from pathlib import Path

# A notebook-cell rerun should replace, not duplicate, the prior server.
if 'server_process' in globals() and server_process.poll() is None:
    print('Stopping the previous Gradio server before restarting...', flush=True)
    server_process.terminate()
    try:
        server_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait(timeout=5)

try:
    from google.colab import userdata as colab_userdata
except ImportError:
    colab_userdata = None

def read_secret(name, *, report_error=False):
    if value := os.environ.get(name):
        return value
    if colab_userdata is None:
        return None
    try:
        return colab_userdata.get(name)
    except Exception as exc:
        if report_error:
            print(f"Colab secret {name!r} is unavailable ({type(exc).__name__}). Check its name and enable notebook access in the Secrets panel.", flush=True)
        return None

SETUP_STAGES = 7
setup_started = time.monotonic()
def stage(number, title, detail):
    elapsed = time.monotonic() - setup_started
    print(f"\n{'=' * 72}\n[{number}/{SETUP_STAGES}] {title}  (elapsed {elapsed / 60:.1f} min)\n{detail}\n{'=' * 72}", flush=True)

stage(1, "Checking the GPU", "This should finish almost immediately.")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Choose Runtime > Change runtime type > GPU, then reconnect.")
subprocess.run(["nvidia-smi"], check=True)

REPO = Path("/content/avtr-1")
PIXI = Path.home() / ".pixi/bin/pixi"
# Engines run from fast ephemeral storage, but a validated copy is kept in Drive.
STORAGE = Path("/content/avtr1_storage")
USE_GOOGLE_DRIVE_ENGINE_CACHE = (read_secret("AVTR_DRIVE_CACHE") or "1").lower() not in {"0", "false", "no"}
DRIVE_ENGINE_CACHE_ROOT = Path("/content/drive/MyDrive/avtr1-engine-cache")
AGENT_VENV = Path("/content/avtr_agent_venv")
AGENT_PYTHON = AGENT_VENV / "bin/python"
configured_repo_url = read_secret("AVTR_REPO_URL")
REPO_URL = configured_repo_url or "https://github.com/RuurdKuiper/avtr-1-colab.git"
# A100 has no native fine-grained FP8 kernels, so use the BF16 checkpoint.
# Set an LLM_MODEL Colab secret to override this (for example the FP8 model on H100).
LLM_MODEL = read_secret("LLM_MODEL") or "Qwen/Qwen3-4B-Instruct-2507"
STT_MODEL = "small"
TTS_MODEL_REPO = "ResembleAI/chatterbox"  # Multilingual V2; shared by English and Dutch.
STT_MODEL_REPO = "Systran/faster-whisper-small"  # Multilingual model; about the same size as small.en.
TTS_ALLOW_PATTERNS = ["ve.pt", "t3_mtl23ls_v2.safetensors", "s3gen.pt", "grapheme_mtl_merged_expanded_v1.json", "conds.pt", "Cangjie5_TC.json"]

stage(2, "Preparing the AVTR-1 checkout", "Cloning on the first run; otherwise pulling updates.")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
if not PIXI.exists():
    subprocess.run(["bash", "-lc", "curl -fsSL https://pixi.sh/install.sh | bash"], check=True)
stage(3, "Installing the locked AVTR environment", "Pixi will reuse packages already cached in this runtime.")
subprocess.run([str(PIXI), "install"], cwd=REPO, check=True)

# TensorRT plans are only reusable with a compatible GPU and TensorRT build.
# Include both, plus the engine-builder sources, in the persistent cache key.
engine_cache_dir = None
engine_cache_key = None
if USE_GOOGLE_DRIVE_ENGINE_CACHE:
    try:
        from google.colab import drive
        print("Mounting Google Drive for the persistent TensorRT engine cache...", flush=True)
        drive.mount("/content/drive")
        runtime_info = json.loads(subprocess.check_output([
            str(PIXI), "run", "python", "-c",
            "import json,torch,tensorrt as trt; p=torch.cuda.get_device_properties(0); print(json.dumps({'gpu':p.name,'cc':[p.major,p.minor],'trt':trt.__version__}))",
        ], cwd=REPO, text=True).strip().splitlines()[-1])
        fingerprint = hashlib.sha256()
        for relative in [
            "pixi.lock",
            "scripts/build_avtr1_engines.py",
            "scripts/build_renderer_engines.py",
            "scripts/build_hubert_engine.py",
            "src/avtr1_renderer/artifact_configs/v1.py",
        ]:
            path = REPO / relative
            fingerprint.update(relative.encode())
            fingerprint.update(path.read_bytes())
        gpu_slug = "-".join(filter(None, "".join(c.lower() if c.isalnum() else "-" for c in runtime_info["gpu"]).split("-")))
        cc = "".join(map(str, runtime_info["cc"]))
        trt_slug = runtime_info["trt"].replace(".", "-")
        engine_cache_key = f"{gpu_slug}-sm{cc}-trt{trt_slug}-{fingerprint.hexdigest()[:12]}"
        engine_cache_dir = DRIVE_ENGINE_CACHE_ROOT / engine_cache_key
        print(f"Engine cache key: {engine_cache_key}", flush=True)
    except Exception as exc:
        engine_cache_dir = None
        print(f"Drive engine cache unavailable ({type(exc).__name__}: {exc}); continuing with local storage.", flush=True)

hf_token = read_secret("HF_TOKEN", report_error=True)
gradio_password = read_secret("GRADIO_PASSWORD")
gradio_username = read_secret("GRADIO_USERNAME") or "avatar"
if not hf_token:
    hf_token = getpass.getpass("Hugging Face read token: ")
if not hf_token:
    raise RuntimeError("HF_TOKEN is required for the gated AVTR-1 weights.")
if not gradio_password:
    gradio_password = secrets.token_urlsafe(12)

env = os.environ.copy()
env.update({
    "HF_TOKEN": hf_token,
    "HF_HUB_TOKEN": hf_token,
    "AVTR1_LOCAL_STORAGE": str(STORAGE),
    "GRADIO_USERNAME": gradio_username,
    "GRADIO_PASSWORD": gradio_password,
    "PYTHONUNBUFFERED": "1",
})
stage(4, "Downloading AVTR artifacts", "Large first-run download; existing files are checked and reused. Hugging Face prints per-file progress below.")
subprocess.run(
    [str(PIXI), "run", "python", "scripts/download_artifacts.py"],
    cwd=REPO, env=env, check=True,
)
engine_root = STORAGE / "main"
speech_engine_dir = engine_root / "speech2motion_runtime_artifacts_cc"
renderer_engine_dir = engine_root / "renderer_runtime_artifacts_cc"
avtr_engines = [speech_engine_dir / "avtr1_encode_fp16.engine", speech_engine_dir / "avtr1_decode_fp16.engine"]
renderer_engines = [renderer_engine_dir / name for name in ["decoder_b5_fp16.engine", "warp_network_b5_fp16.engine", "modnet_b5_fp16.engine", "stitch_network_b5_fp16.engine"]]
hubert_engine = speech_engine_dir / "hubert_lbs_fp16.engine"
normalizer = engine_root / "avtr1_normalizer.safetensors"
engine_cache_files = [*avtr_engines, *renderer_engines, hubert_engine, normalizer]

# Restore only a complete cache whose manifest and file sizes match. Files are
# copied to /content because loading large TensorRT plans directly from Drive is slow.
if engine_cache_dir is not None:
    manifest_path = engine_cache_dir / "manifest.json"
    try:
        manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
        expected = {str(path.relative_to(STORAGE)) for path in engine_cache_files}
        cached_files = manifest.get("files", {})
        cache_complete = (
            manifest.get("cache_key") == engine_cache_key
            and set(cached_files) == expected
            and all((engine_cache_dir / rel).is_file()
                    and (engine_cache_dir / rel).stat().st_size == size
                    and size > 0 for rel, size in cached_files.items())
        )
        if cache_complete:
            restore_started = time.monotonic()
            total_bytes = 0
            for rel, size in cached_files.items():
                destination = STORAGE / rel
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(engine_cache_dir / rel, destination)
                total_bytes += size
            print(f"Restored {len(cached_files)} TensorRT cache files ({total_bytes / 1e9:.2f} GB) from Drive in {time.monotonic() - restore_started:.1f}s.", flush=True)
        else:
            print("No complete compatible Drive engine cache found; this runtime will build it once.", flush=True)
    except Exception as exc:
        print(f"Could not restore the Drive engine cache ({type(exc).__name__}: {exc}); rebuilding locally.", flush=True)
stage(5, "Preparing TensorRT engines", "AVTR motion, renderer, and HuBERT engines are required. TensorRT may appear quiet while optimizing a graph; keep the cell running.")
if not all(path.is_file() and path.stat().st_size > 0 for path in [*avtr_engines, normalizer]):
    print("Building the two AVTR motion engines.", flush=True)
    subprocess.run([str(PIXI), "run", "build-trt-engines-avtr1"], cwd=REPO, env=env, check=True)
else:
    print("Cache hit: reusing both AVTR motion engines.", flush=True)
if not all(path.is_file() and path.stat().st_size > 0 for path in renderer_engines):
    print("Building the four renderer engines, including the required GridSample3D warp engine.", flush=True)
    subprocess.run([str(PIXI), "run", "build-trt-engines-renderer"], cwd=REPO, env=env, check=True)
else:
    print("Cache hit: reusing all four renderer engines.", flush=True)
if not hubert_engine.is_file() or hubert_engine.stat().st_size == 0:
    print("Building the required HuBERT TensorRT engine.", flush=True)
    subprocess.run([str(PIXI), "run", "build-trt-engines-hubert"], cwd=REPO, env=env, check=True)
else:
    print("Cache hit: reusing the HuBERT TensorRT engine.", flush=True)

# Publish the cache only after every required output exists. The manifest is
# written last, so an interrupted copy is ignored on the next startup.
if engine_cache_dir is not None:
    missing_cache_outputs = [path for path in engine_cache_files if not path.is_file() or path.stat().st_size == 0]
    if missing_cache_outputs:
        raise RuntimeError(f"TensorRT build finished without required outputs: {missing_cache_outputs}")
    try:
        save_started = time.monotonic()
        engine_cache_dir.mkdir(parents=True, exist_ok=True)
        cache_manifest = {"cache_key": engine_cache_key, "files": {}}
        for source in engine_cache_files:
            rel = source.relative_to(STORAGE)
            destination = engine_cache_dir / rel
            destination.parent.mkdir(parents=True, exist_ok=True)
            if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
                shutil.copy2(source, destination)
            cache_manifest["files"][str(rel)] = source.stat().st_size
        temporary_manifest = engine_cache_dir / "manifest.json.tmp"
        temporary_manifest.write_text(json.dumps(cache_manifest, indent=2))
        os.replace(temporary_manifest, engine_cache_dir / "manifest.json")
        print(f"Saved validated TensorRT engine cache to Drive in {time.monotonic() - save_started:.1f}s.", flush=True)
    except Exception as exc:
        print(f"Warning: could not save the Drive engine cache ({type(exc).__name__}: {exc}). The current runtime can still continue.", flush=True)

stage(6, "Installing the conversational stack", f"Creating an isolated environment, fixing Perth's runtime dependency, and downloading {LLM_MODEL}, {TTS_MODEL_REPO}, and {STT_MODEL_REPO}. Existing caches are reused.")
venv_ready = AGENT_PYTHON.exists() and subprocess.run(
    [str(AGENT_PYTHON), "-m", "pip", "--version"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
).returncode == 0
if not venv_ready:
    if AGENT_VENV.exists():
        print("Removing the incomplete assistant environment left by a previous attempt.", flush=True)
        shutil.rmtree(AGENT_VENV)
    print("Creating the assistant environment with AVTR's managed Python 3.12.", flush=True)
    subprocess.run(
        [str(PIXI), "run", "python", "-m", "venv", str(AGENT_VENV)],
        cwd=REPO, env=env, check=True,
    )
subprocess.run([str(AGENT_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools==80.9.0", "wheel"], check=True)
subprocess.run([
    str(AGENT_PYTHON), "-m", "pip", "install",
    "chatterbox-tts==0.1.7", "faster-whisper>=1.1,<2", "accelerate>=1,<2",
], check=True)
subprocess.run([
    str(AGENT_PYTHON), "-c",
    "import perth; assert perth.PerthImplicitWatermarker is not None, 'Perth watermarker import failed'",
], env=env, check=True)
for model_repo in [LLM_MODEL, TTS_MODEL_REPO, STT_MODEL_REPO]:
    print(f"Downloading/checking model cache: {model_repo}", flush=True)
    subprocess.run([
        str(AGENT_PYTHON), "-c",
        "import json,sys; from huggingface_hub import snapshot_download; snapshot_download(repo_id=sys.argv[1], allow_patterns=json.loads(sys.argv[2]) or None)",
        model_repo, json.dumps(TTS_ALLOW_PATTERNS if model_repo == TTS_MODEL_REPO else []),
    ], env=env, check=True)

# The server is regular version-controlled source in the cloned repository.
app_path = REPO / "scripts/gradio_colab_app.py"
if not app_path.is_file():
    raise RuntimeError(
        f"Missing {app_path}. Check AVTR_REPO_URL and make sure it points to "
        "the repository/branch containing the Colab server code."
    )
print(f"Using version-controlled server script: {app_path}", flush=True)

stage(7, "Preloading models and launching Gradio", "The server now loads Qwen, Whisper, Chatterbox, and the persistent AVTR renderer. Their individual timings print below; keep this cell running while using the interface.")
print(f"Bootstrap time before model warmup: {(time.monotonic() - setup_started) / 60:.1f} minutes", flush=True)
print("Open the gradio.live URL when it appears after model warmup.", flush=True)
print(f"Login: {gradio_username} / {gradio_password}\n", flush=True)
server_process = subprocess.Popen([
    str(AGENT_PYTHON), str(app_path),
    "--repo", str(REPO),
    "--pixi", str(PIXI),
    "--storage", str(STORAGE),
    "--llm-model", LLM_MODEL,
    "--stt-model", STT_MODEL,
], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert server_process.stdout is not None
for line in server_process.stdout:
    print(line, end="", flush=True)
server_returncode = server_process.wait()
if server_returncode:
    raise RuntimeError(f"Gradio server exited with status {server_returncode}; see its full exception immediately above.")
